# Порівняння методів автоматичної оцінки

Рядок таблиці — одна відповідь системи. Стовпці — режим RAG, оцінки методів і ручна розмітка.
Кореляції рахуються лише там, де для пари є спільні непорожні значення. Порожня база не малює фіктивних результатів: спочатку потрібні `run_experiments.py` і імпорт розмітки.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import roc_auc_score

from config import FIGURES_DIR
from db import to_dataframe

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
df = to_dataframe()
df.head()


## Що порівнюємо

Вищі значення метрик нижче означають кращу відповідь. Для AUC за міткою «галюцинація» їх змінюють знаком: велика автоматична оцінка має відповідати відсутності галюцинації.
`decision__missing_count` навпаки зростає, коли для рішення бракує елементів, тому його не включають у цей список.


In [ ]:
QUALITY = [
    "rouge__rougeL",
    "bertscore__f1",
    "nli__support_rate",
    "judge_plain__score",
    "judge_plain_gpt4o__score",
    "judge_rubric__faithfulness",
    "judge_rubric__completeness",
    "judge_rubric__source_alignment",
    "judge_rubric__decision_usefulness",
    "judge_rubric_gpt4o__faithfulness",
    "ragas__faithfulness",
    "ragas__answer_relevancy",
    "ragas__context_precision",
    "ragas__context_recall",
    "decision__usefulness",
    "pairwise__win_rate",
    "selfcheck__agreement",
]

EXPERT = {
    "expert_faithfulness": ["nli__support_rate", "judge_rubric__faithfulness", "judge_rubric_gpt4o__faithfulness", "ragas__faithfulness", "rouge__rougeL", "bertscore__f1"],
    "expert_completeness": ["judge_rubric__completeness", "ragas__answer_relevancy", "rouge__rougeL", "bertscore__f1"],
    "expert_usefulness": ["judge_rubric__decision_usefulness", "decision__usefulness", "pairwise__win_rate"],
    "expert_trust": QUALITY,
}

def paired(frame, left, right):
    cols = frame[[left, right]].dropna()
    return cols

def correlation_table(frame):
    rows = []
    for expert, methods in EXPERT.items():
        if expert not in frame.columns:
            continue
        for method in methods:
            if method not in frame.columns:
                continue
            cols = paired(frame, method, expert)
            if len(cols) < 5 or cols[method].nunique() < 2 or cols[expert].nunique() < 2:
                continue
            spearman = spearmanr(cols[method], cols[expert])
            kendall = kendalltau(cols[method], cols[expert])
            rows.append({
                "expert": expert,
                "method": method,
                "n": len(cols),
                "spearman": spearman.correlation,
                "spearman_p": spearman.pvalue,
                "kendall": kendall.correlation,
                "kendall_p": kendall.pvalue,
            })
    return pd.DataFrame(rows)

corr = correlation_table(df)
corr.sort_values("spearman", ascending=False) if not corr.empty else corr


## ROC-AUC: чи відрізняє метод галюцинацію

Мітка експерта `expert_hallucination` дорівнює 1, якщо у відповіді є вигаданий факт. У `roc_auc_score` передається мінус автоматичної оцінки, бо вища якість має означати меншу ймовірність галюцинації.


In [ ]:
def hallucination_auc(frame):
    if "expert_hallucination" not in frame.columns:
        return pd.DataFrame()
    rows = []
    for method in QUALITY:
        if method not in frame.columns:
            continue
        cols = paired(frame, method, "expert_hallucination")
        if len(cols) < 5 or cols["expert_hallucination"].nunique() < 2:
            continue
        rows.append({
            "method": method,
            "n": len(cols),
            "auc": roc_auc_score(cols["expert_hallucination"], -cols[method]),
        })
    return pd.DataFrame(rows).sort_values("auc", ascending=False)

auc = hallucination_auc(df)
auc


## Ті самі показники всередині кожного режиму

Окремий режим потрібен, щоб побачити, чи метод працює лише на відповідях без пошуку або лише на локальних документах.


In [ ]:
def by_mode(frame):
    rows = []
    if "expert_faithfulness" not in frame.columns:
        return pd.DataFrame()
    for mode, group in frame.groupby("mode"):
        table = correlation_table(group)
        if table.empty:
            continue
        part = table[table["expert"] == "expert_faithfulness"].copy()
        part["mode"] = mode
        rows.append(part)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

mode_corr = by_mode(df)
mode_corr


## Розкид судді, вартість і час

`__std` — стандартне відхилення повторних прогонів однієї пари відповідь–критерій. Воно з'являється, якщо експеримент запущено з `--judge-repeats` більше за 1. Вартість і час усереднені за прогонами методу.


In [ ]:
def judge_spread(frame):
    columns = [col for col in frame.columns if "__std" in col and "judge_" in col]
    if not columns:
        return pd.Series(dtype=float)
    return frame[columns].mean().sort_values(ascending=False)

def cost_and_time(frame):
    rows = []
    methods = sorted({col.split("__cost")[0] for col in frame.columns if col.endswith("__cost")})
    for method in methods:
        rows.append({
            "method": method,
            "mean_cost_usd": frame[f"{method}__cost"].mean() if f"{method}__cost" in frame else np.nan,
            "mean_latency_s": frame[f"{method}__latency"].mean() if f"{method}__latency" in frame else np.nan,
        })
    return pd.DataFrame(rows).sort_values("mean_cost_usd", ascending=False)

spread = judge_spread(df)
cost = cost_and_time(df)
spread, cost


## Бутстреп довірчих інтервалів

Для кореляції Спірмена з `expert_faithfulness` вибірка з поверненням будується 1000 разів. Інтервал — 2,5 і 97,5 перцентилі. Якщо пар менше ніж 8, інтервал не рахується.


In [ ]:
def bootstrap_spearman(frame, method, expert="expert_faithfulness", n_boot=1000, seed=42):
    if method not in frame.columns or expert not in frame.columns:
        return None
    cols = paired(frame, method, expert)
    if len(cols) < 8:
        return None
    rng = np.random.default_rng(seed)
    stats = []
    values = cols.to_numpy()
    for _ in range(n_boot):
        sample = values[rng.choice(len(values), size=len(values), replace=True)]
        if len(np.unique(sample[:, 0])) < 2 or len(np.unique(sample[:, 1])) < 2:
            continue
        stats.append(spearmanr(sample[:, 0], sample[:, 1]).correlation)
    if not stats:
        return None
    arr = np.asarray(stats, dtype=float)
    return {
        "method": method,
        "spearman": float(spearmanr(cols[method], cols[expert]).correlation),
        "ci_low": float(np.percentile(arr, 2.5)),
        "ci_high": float(np.percentile(arr, 97.5)),
        "n": int(len(cols)),
    }

intervals = pd.DataFrame([
    item for item in (
        bootstrap_spearman(df, method)
        for method in QUALITY
    ) if item
])
intervals


## Узгодженість двох розмітників

Другий експерт лежить у стовпцях `expert2_*`. Кореляція між розмітниками показує, наскільки стабільний еталон, з яким порівнюють автоматичні методи.


In [ ]:
def annotator_agreement(frame):
    rows = []
    for field in ("faithfulness", "completeness", "usefulness", "trust", "hallucination"):
        left, right = f"expert_{field}", f"expert2_{field}"
        if left not in frame.columns or right not in frame.columns:
            continue
        cols = paired(frame, left, right)
        if len(cols) < 5 or cols[left].nunique() < 2 or cols[right].nunique() < 2:
            continue
        stat = spearmanr(cols[left], cols[right])
        rows.append({"field": field, "n": len(cols), "spearman": stat.correlation, "p": stat.pvalue})
    return pd.DataFrame(rows)

agreement = annotator_agreement(df)
agreement


## Графіки

Стовпчики — кореляція з достовірністю експерта. Діаграма розмаху — як NLI змінюється за режимом пошуку, якщо ця оцінка вже порахована. Точкова діаграма — автоматична достовірність рубрики проти експерта.


In [ ]:
if corr.empty:
    print("Немає пар метод–експерт. Заповніть експеримент і розмітку, потім запустіть зошит знову.")
else:
    faith = corr[corr["expert"] == "expert_faithfulness"].sort_values("spearman")
    if not faith.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(faith["method"], faith["spearman"])
        ax.set_xlabel("Спірмен з expert_faithfulness")
        ax.set_title("Узгодженість автоматичних методів з експертом")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "spearman_faithfulness.png", dpi=150)
        plt.show()

if "nli__support_rate" in df.columns and df["mode"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(7, 4))
    df.boxplot(column="nli__support_rate", by="mode", ax=ax)
    ax.set_title("NLI за режимом")
    ax.set_ylabel("частка підтверджених тверджень")
    plt.suptitle("")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "nli_by_mode.png", dpi=150)
    plt.show()

if {"judge_rubric__faithfulness", "expert_faithfulness"} <= set(df.columns):
    cols = paired(df, "judge_rubric__faithfulness", "expert_faithfulness")
    if not cols.empty:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(cols["expert_faithfulness"], cols["judge_rubric__faithfulness"], alpha=0.7)
        ax.set_xlabel("Експерт")
        ax.set_ylabel("Суддя за рубрикою")
        ax.set_title("Достовірність")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "rubric_vs_expert.png", dpi=150)
        plt.show()
